In [1]:

# ============================================================
# MACHINE LEARNING FORECASTING MODELS
# ============================================================

import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.model_selection import (
    GridSearchCV,
    TimeSeriesSplit
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

cwd = os.getcwd()

PROJECT_ROOT = (
    os.path.dirname(cwd)
    if os.path.basename(cwd).lower() == "notebooks"
    else cwd
)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "machine_learning")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load forecasting dataset
# ------------------------------------------------------------

dataset_path = os.path.join(
    OUTPUT_DIR,
    "forecasting_framework_dataset.csv"
)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        "forecasting_framework_dataset.csv not found. "
        "Run the forecasting framework notebook first."
    )

df = pd.read_csv(dataset_path)

df.columns = df.columns.astype(str).str.strip().str.lower()

# ------------------------------------------------------------
# Identify date and target columns
# ------------------------------------------------------------

date_candidates = [
    "date",
    "report_date",
    "observation_date",
    "snapshot_date"
]

date_col = next(
    (column for column in date_candidates if column in df.columns),
    None
)

if date_col is None:
    raise ValueError("Date column not found in forecasting dataset.")

target_col = None

statistical_config_path = os.path.join(
    OUTPUT_DIR,
    "statistical_forecasting_configuration.json"
)

if os.path.exists(statistical_config_path):

    with open(statistical_config_path, "r") as file:
        previous_config = json.load(file)

    configured_target = previous_config.get("target_variable")

    if configured_target in df.columns:
        target_col = configured_target

if target_col is None:

    target_candidates = [
        "children_in_hhs_care",
        "total_children",
        "total_care_load",
        "care_load",
        "placement_demand"
    ]

    target_col = next(
        (column for column in target_candidates if column in df.columns),
        None
    )

if target_col is None:

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    numeric_columns = [
        column for column in numeric_columns
        if column != date_col
        and not any(
            term in column
            for term in ["lag_", "rolling_", "moving_average"]
        )
    ]

    if len(numeric_columns) == 1:
        target_col = numeric_columns[0]
    else:
        raise ValueError(
            "Unable to identify the forecasting target safely. "
            f"Available numeric columns: {numeric_columns}"
        )

df[date_col] = pd.to_datetime(
    df[date_col],
    errors="coerce"
)

df[target_col] = pd.to_numeric(
    df[target_col],
    errors="coerce"
)

df = (
    df.dropna(subset=[date_col, target_col])
    .sort_values(date_col)
    .drop_duplicates(subset=[date_col])
    .reset_index(drop=True)
)

print("Dataset loaded successfully.")
print("Date column:", date_col)
print("Target column:", target_col)
print("Observations:", len(df))

# ------------------------------------------------------------
# Load chronological split boundaries
# ------------------------------------------------------------

split_path = os.path.join(
    OUTPUT_DIR,
    "chronological_split_summary.csv"
)

if not os.path.exists(split_path):
    raise FileNotFoundError(
        "chronological_split_summary.csv not found."
    )

split_summary = pd.read_csv(split_path)

def get_split_date(dataset_name):

    result = split_summary.loc[
        split_summary["Dataset"].astype(str).str.lower()
        == dataset_name.lower(),
        "End_Date"
    ]

    if result.empty:
        raise ValueError(
            f"End date for {dataset_name} not found."
        )

    return pd.to_datetime(result.iloc[0])


train_end = get_split_date("Training")
validation_end = get_split_date("Validation")

# ------------------------------------------------------------
# Create chronological partitions
# ------------------------------------------------------------

df = df.reset_index(drop=True)

train_mask = df[date_col] <= train_end

validation_mask = (
    (df[date_col] > train_end)
    & (df[date_col] <= validation_end)
)

test_mask = df[date_col] > validation_end

if not train_mask.any() or not validation_mask.any() or not test_mask.any():
    raise ValueError(
        "Training, validation, or testing partition is empty."
    )

print("\nChronological boundaries:")
print("Training observations:", train_mask.sum())
print("Validation observations:", validation_mask.sum())
print("Testing observations:", test_mask.sum())

# ------------------------------------------------------------
# Create forecasting features
# ------------------------------------------------------------

def create_features(data):

    features = pd.DataFrame(index=data.index)

    values = data[target_col].astype(float)

    lag_periods = [1, 2, 3, 7, 14, 30]

    for lag in lag_periods:
        features[f"lag_{lag}"] = values.shift(lag)

    features["rolling_mean_7"] = (
        values.shift(1).rolling(7).mean()
    )

    features["rolling_mean_14"] = (
        values.shift(1).rolling(14).mean()
    )

    features["rolling_mean_30"] = (
        values.shift(1).rolling(30).mean()
    )

    features["rolling_std_7"] = (
        values.shift(1).rolling(7).std()
    )

    features["rolling_std_14"] = (
        values.shift(1).rolling(14).std()
    )

    features["rolling_min_7"] = (
        values.shift(1).rolling(7).min()
    )

    features["rolling_max_7"] = (
        values.shift(1).rolling(7).max()
    )

    dates = data[date_col]

    features["day_of_week"] = dates.dt.dayofweek
    features["day_of_month"] = dates.dt.day
    features["month"] = dates.dt.month
    features["quarter"] = dates.dt.quarter
    features["year"] = dates.dt.year
    features["day_of_year"] = dates.dt.dayofyear

    features["month_sin"] = np.sin(
        2 * np.pi * dates.dt.month / 12
    )

    features["month_cos"] = np.cos(
        2 * np.pi * dates.dt.month / 12
    )

    features["day_of_year_sin"] = np.sin(
        2 * np.pi * dates.dt.dayofyear / 365.25
    )

    features["day_of_year_cos"] = np.cos(
        2 * np.pi * dates.dt.dayofyear / 365.25
    )

    return features


feature_df = create_features(df)

# ------------------------------------------------------------
# Prepare machine learning training dataset
# ------------------------------------------------------------

modeling_data = pd.concat(
    [
        df[[date_col, target_col]],
        feature_df
    ],
    axis=1
)

modeling_data = modeling_data.replace(
    [np.inf, -np.inf],
    np.nan
)

modeling_data = modeling_data.dropna().reset_index(drop=True)

feature_columns = feature_df.columns.tolist()

if len(modeling_data) < 40:
    raise ValueError(
        "Insufficient observations after feature preparation."
    )

X_all = modeling_data[feature_columns]

print("\nFeature preparation completed.")
print("Number of features:", len(feature_columns))
print("Modeling observations:", len(modeling_data))

# ------------------------------------------------------------
# Define forecast horizons
# ------------------------------------------------------------

horizons = {
    "1_day": 1,
    "7_day": 7,
    "14_day": 14,
    "30_day": 30
}

framework_config_path = os.path.join(
    OUTPUT_DIR,
    "forecasting_configuration.json"
)

if os.path.exists(framework_config_path):

    with open(framework_config_path, "r") as file:
        framework_config = json.load(file)

    saved_horizons = framework_config.get(
        "forecast_horizon_steps", {}
    )

    for horizon_name in horizons:

        if horizon_name in saved_horizons:
            horizons[horizon_name] = int(
                saved_horizons[horizon_name]
            )

print("\nForecast horizons:", horizons)

# ------------------------------------------------------------
# Define model parameters and tuning grids
# ------------------------------------------------------------

model_grids = {

    "Random_Forest": {

        "estimator": RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),

        "parameters": {
            "n_estimators": [100, 200],
            "max_depth": [5, 10, None],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2]
        }

    },

    "Gradient_Boosting": {

        "estimator": GradientBoostingRegressor(
            random_state=42
        ),

        "parameters": {
            "n_estimators": [100, 200],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [2, 3],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2]
        }

    }

}

# ------------------------------------------------------------
# Prepare direct multi-horizon datasets
# ------------------------------------------------------------

direct_datasets = {}

for horizon_name, steps in horizons.items():

    direct_data = modeling_data.copy()

    direct_data["future_target"] = (
        direct_data[target_col].shift(-steps)
    )

    direct_data["future_date"] = (
        direct_data[date_col].shift(-steps)
    )

    direct_data = direct_data.dropna(
        subset=["future_target", "future_date"]
    ).copy()

    direct_datasets[horizon_name] = direct_data

# ------------------------------------------------------------
# Tune and train direct forecasting models
# ------------------------------------------------------------

trained_models = {}
tuning_results = []
validation_predictions = []
test_predictions = []
feature_importance_results = []

for horizon_name, steps in horizons.items():

    direct_data = direct_datasets[horizon_name]

    train_data = direct_data[
        direct_data["future_date"] <= train_end
    ].copy()

    validation_data = direct_data[
        (direct_data["future_date"] > train_end)
        & (direct_data["future_date"] <= validation_end)
    ].copy()

    test_data = direct_data[
        direct_data["future_date"] > validation_end
    ].copy()

    if len(train_data) < 20:
        print(
            f"Skipping {horizon_name}: insufficient training samples."
        )
        continue

    X_train = train_data[feature_columns]
    y_train = train_data["future_target"]

    X_validation = validation_data[feature_columns]
    y_validation = validation_data["future_target"]

    X_test = test_data[feature_columns]
    y_test = test_data["future_target"]

    n_splits = min(5, max(2, len(train_data) // 10))

    time_series_cv = TimeSeriesSplit(
        n_splits=n_splits
    )

    for model_name, settings in model_grids.items():

        print(
            f"\nTraining {model_name} for {horizon_name}"
        )

        search = GridSearchCV(
            estimator=settings["estimator"],
            param_grid=settings["parameters"],
            scoring="neg_mean_absolute_error",
            cv=time_series_cv,
            n_jobs=-1,
            refit=True,
            error_score="raise"
        )

        search.fit(X_train, y_train)

        best_model = search.best_estimator_

        trained_models[
            f"{model_name}_{horizon_name}"
        ] = best_model

        tuning_results.append({

            "Model": model_name,
            "Horizon": horizon_name,
            "Best_CV_MAE": -search.best_score_,
            "Best_Parameters": str(search.best_params_)

        })

        # Validation predictions

        if len(validation_data) > 0:

            validation_pred = best_model.predict(
                X_validation
            )

            for i in range(len(validation_data)):

                validation_predictions.append({

                    "Model": model_name,
                    "Forecasting_Method": "Direct",
                    "Horizon": horizon_name,
                    "Forecast_Origin": validation_data.iloc[i][date_col],
                    "Actual_Target_Date": validation_data.iloc[i]["future_date"],
                    "Actual": float(y_validation.iloc[i]),
                    "Predicted": float(validation_pred[i])

                })

        # Testing predictions

        if len(test_data) > 0:

            test_pred = best_model.predict(X_test)

            for i in range(len(test_data)):

                test_predictions.append({

                    "Model": model_name,
                    "Forecasting_Method": "Direct",
                    "Horizon": horizon_name,
                    "Forecast_Origin": test_data.iloc[i][date_col],
                    "Actual_Target_Date": test_data.iloc[i]["future_date"],
                    "Actual": float(y_test.iloc[i]),
                    "Predicted": float(test_pred[i])

                })

        # Feature importance

        if hasattr(best_model, "feature_importances_"):

            importance = best_model.feature_importances_

            for feature, value in zip(
                feature_columns,
                importance
            ):

                feature_importance_results.append({

                    "Model": model_name,
                    "Horizon": horizon_name,
                    "Feature": feature,
                    "Importance": float(value)

                })

        print("Best parameters:", search.best_params_)
        print("Best CV MAE:", -search.best_score_)

# ------------------------------------------------------------
# Prepare recursive one-step training dataset
# ------------------------------------------------------------

recursive_data = modeling_data.copy()

recursive_data["future_target"] = (
    recursive_data[target_col].shift(-1)
)

recursive_data = recursive_data.dropna(
    subset=["future_target"]
).copy()

recursive_train = recursive_data[
    recursive_data[date_col] <= train_end
].copy()

if len(recursive_train) < 20:
    raise ValueError(
        "Insufficient observations for recursive forecasting."
    )

X_recursive_train = recursive_train[feature_columns]

y_recursive_train = recursive_train["future_target"]

# ------------------------------------------------------------
# Tune recursive one-step models
# ------------------------------------------------------------

recursive_models = {}

for model_name, settings in model_grids.items():

    print(f"\nTraining recursive {model_name}")

    recursive_search = GridSearchCV(
        estimator=settings["estimator"],
        param_grid=settings["parameters"],
        scoring="neg_mean_absolute_error",
        cv=TimeSeriesSplit(
            n_splits=min(
                5,
                max(2, len(recursive_train) // 10)
            )
        ),
        n_jobs=-1,
        refit=True,
        error_score="raise"
    )

    recursive_search.fit(
        X_recursive_train,
        y_recursive_train
    )

    recursive_models[model_name] = (
        recursive_search.best_estimator_
    )

    tuning_results.append({

        "Model": model_name,
        "Horizon": "Recursive_One_Step",
        "Best_CV_MAE": -recursive_search.best_score_,
        "Best_Parameters": str(
            recursive_search.best_params_
        )

    })

    print("Best parameters:", recursive_search.best_params_)

# ------------------------------------------------------------
# Build recursive forecasting features
# ------------------------------------------------------------

def build_next_feature(history, forecast_date):

    history = list(history)

    feature_values = {}

    for lag in [1, 2, 3, 7, 14, 30]:

        feature_values[f"lag_{lag}"] = (
            history[-lag]
            if len(history) >= lag
            else np.nan
        )

    for window in [7, 14, 30]:

        recent = np.asarray(history[-window:])

        feature_values[f"rolling_mean_{window}"] = (
            np.mean(recent)
            if len(recent) >= window
            else np.nan
        )

    for window in [7, 14]:

        recent = np.asarray(history[-window:])

        feature_values[f"rolling_std_{window}"] = (
            np.std(recent, ddof=1)
            if len(recent) >= window
            else np.nan
        )

    recent_7 = np.asarray(history[-7:])

    feature_values["rolling_min_7"] = np.min(recent_7)
    feature_values["rolling_max_7"] = np.max(recent_7)

    feature_values["day_of_week"] = forecast_date.dayofweek
    feature_values["day_of_month"] = forecast_date.day
    feature_values["month"] = forecast_date.month
    feature_values["quarter"] = forecast_date.quarter
    feature_values["year"] = forecast_date.year
    feature_values["day_of_year"] = forecast_date.dayofyear

    feature_values["month_sin"] = np.sin(
        2 * np.pi * forecast_date.month / 12
    )

    feature_values["month_cos"] = np.cos(
        2 * np.pi * forecast_date.month / 12
    )

    feature_values["day_of_year_sin"] = np.sin(
        2 * np.pi * forecast_date.dayofyear / 365.25
    )

    feature_values["day_of_year_cos"] = np.cos(
        2 * np.pi * forecast_date.dayofyear / 365.25
    )

    return pd.DataFrame(
        [feature_values],
        columns=feature_columns
    )

# ------------------------------------------------------------
# Generate recursive validation and testing forecasts
# ------------------------------------------------------------

recursive_predictions = []

for model_name, model in recursive_models.items():

    for partition_name, partition_end in [
        ("Validation", train_end),
        ("Testing", validation_end)
    ]:

        historical_data = df[
            df[date_col] <= partition_end
        ].copy()

        future_data = df[
            df[date_col] > partition_end
        ].copy()

        if partition_name == "Validation":

            future_data = future_data[
                future_data[date_col] <= validation_end
            ]

        if future_data.empty:
            continue

        history = historical_data[
            target_col
        ].astype(float).tolist()

        last_date = historical_data[date_col].iloc[-1]

        future_dates = future_data[
            date_col
        ].tolist()

        for position, actual_row in enumerate(
            future_data.itertuples(index=False)
        ):

            forecast_date = future_dates[position]

            # Build features using observations available
            # before the forecast date.

            X_next = build_next_feature(
                history,
                forecast_date
            )

            if X_next.isna().any().any():
                break

            predicted = float(
                model.predict(X_next)[0]
            )

            actual = float(
                future_data.iloc[position][target_col]
            )

            recursive_predictions.append({

                "Model": model_name,
                "Forecasting_Method": "Recursive",
                "Partition": partition_name,
                "Horizon": "Sequential_One_Step",
                "Forecast_Origin": last_date,
                "Actual_Target_Date": forecast_date,
                "Actual": actual,
                "Predicted": predicted

            })

            # Recursive forecasting feeds predictions
            # into subsequent forecasting steps.

            history.append(predicted)

# ------------------------------------------------------------
# Combine direct and recursive predictions
# ------------------------------------------------------------

validation_predictions = pd.DataFrame(
    validation_predictions
)

test_predictions = pd.DataFrame(
    test_predictions
)

recursive_predictions = pd.DataFrame(
    recursive_predictions
)

recursive_validation = (
    recursive_predictions[
        recursive_predictions["Partition"] == "Validation"
    ].drop(columns=["Partition"])
    if not recursive_predictions.empty
    else pd.DataFrame()
)

recursive_test = (
    recursive_predictions[
        recursive_predictions["Partition"] == "Testing"
    ].drop(columns=["Partition"])
    if not recursive_predictions.empty
    else pd.DataFrame()
)

all_validation_predictions = pd.concat(
    [
        validation_predictions,
        recursive_validation
    ],
    ignore_index=True
)

all_test_predictions = pd.concat(
    [
        test_predictions,
        recursive_test
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Evaluate prediction performance
# ------------------------------------------------------------

def calculate_metrics(predictions):

    results = []

    if predictions.empty:
        return pd.DataFrame()

    for (model_name, method, horizon), group in predictions.groupby(
        ["Model", "Forecasting_Method", "Horizon"]
    ):

        actual = group["Actual"].values
        predicted = group["Predicted"].values

        nonzero = actual != 0

        results.append({

            "Model": model_name,
            "Forecasting_Method": method,
            "Horizon": horizon,
            "Observations": len(group),
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(
                mean_squared_error(actual, predicted)
            ),
            "MAPE_Percent": (
                mean_absolute_percentage_error(
                    actual[nonzero],
                    predicted[nonzero]
                ) * 100
                if nonzero.any()
                else np.nan
            ),
            "Bias": np.mean(predicted - actual)

        })

    return pd.DataFrame(results)


validation_metrics = calculate_metrics(
    all_validation_predictions
)

test_metrics = calculate_metrics(
    all_test_predictions
)

# ------------------------------------------------------------
# Analyze prediction errors
# ------------------------------------------------------------

def create_error_analysis(predictions):

    if predictions.empty:
        return pd.DataFrame()

    errors = predictions.copy()

    errors["Error"] = (
        errors["Predicted"] - errors["Actual"]
    )

    errors["Absolute_Error"] = errors["Error"].abs()

    errors["Percentage_Error"] = np.where(
        errors["Actual"] != 0,
        errors["Error"] / errors["Actual"] * 100,
        np.nan
    )

    errors["Error_Direction"] = np.select(
        [
            errors["Error"] > 0,
            errors["Error"] < 0
        ],
        [
            "Overprediction",
            "Underprediction"
        ],
        default="Exact Prediction"
    )

    return errors


validation_errors = create_error_analysis(
    all_validation_predictions
)

test_errors = create_error_analysis(
    all_test_predictions
)

# ------------------------------------------------------------
# Summarize feature importance
# ------------------------------------------------------------

feature_importance_df = pd.DataFrame(
    feature_importance_results
)

if not feature_importance_df.empty:

    feature_importance_summary = (
        feature_importance_df
        .groupby("Feature", as_index=False)["Importance"]
        .mean()
        .sort_values("Importance", ascending=False)
    )

else:

    feature_importance_summary = pd.DataFrame(
        columns=["Feature", "Importance"]
    )

# ------------------------------------------------------------
# Save model tuning results
# ------------------------------------------------------------

pd.DataFrame(tuning_results).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_hyperparameter_results.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save validation and testing predictions
# ------------------------------------------------------------

all_validation_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_validation_predictions.csv"
    ),
    index=False
)

all_test_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_test_predictions.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save model evaluation metrics
# ------------------------------------------------------------

validation_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_validation_metrics.csv"
    ),
    index=False
)

test_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_test_metrics.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save feature importance results
# ------------------------------------------------------------

feature_importance_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_feature_importance.csv"
    ),
    index=False
)

feature_importance_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_feature_importance_summary.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save prediction error analysis
# ------------------------------------------------------------

validation_errors.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_validation_errors.csv"
    ),
    index=False
)

test_errors.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_test_errors.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save recursive forecasting predictions
# ------------------------------------------------------------

recursive_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_recursive_predictions.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save trained model artifacts
# ------------------------------------------------------------

for model_name, model in trained_models.items():

    joblib.dump(
        model,
        os.path.join(
            MODEL_DIR,
            f"{model_name.lower()}.joblib"
        )
    )

for model_name, model in recursive_models.items():

    joblib.dump(
        model,
        os.path.join(
            MODEL_DIR,
            f"{model_name.lower()}_recursive.joblib"
        )
    )

# ------------------------------------------------------------
# Save model configuration
# ------------------------------------------------------------

machine_learning_config = {

    "date_column": date_col,
    "target_variable": target_col,
    "features": feature_columns,
    "forecast_horizons": horizons,
    "training_end_date": str(train_end),
    "validation_end_date": str(validation_end),
    "training_observations": int(train_mask.sum()),
    "validation_observations": int(validation_mask.sum()),
    "testing_observations": int(test_mask.sum()),
    "direct_models": list(trained_models.keys()),
    "recursive_models": list(recursive_models.keys()),
    "tuning_method": "GridSearchCV with TimeSeriesSplit",
    "scoring_metric": "Negative Mean Absolute Error",
    "random_state": 42

}

with open(
    os.path.join(
        OUTPUT_DIR,
        "machine_learning_configuration.json"
    ),
    "w"
) as file:

    json.dump(
        machine_learning_config,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print("\n==============================================")
print("MACHINE LEARNING FORECASTING COMPLETED")
print("==============================================")

print("\nTrained direct models:")

for name in trained_models:
    print("-", name)

print("\nTrained recursive models:")

for name in recursive_models:
    print("-", name)

print("\nVALIDATION METRICS")
print(
    validation_metrics.to_string(index=False)
    if not validation_metrics.empty
    else "No validation metrics available."
)

print("\nTEST METRICS")
print(
    test_metrics.to_string(index=False)
    if not test_metrics.empty
    else "No test metrics available."
)

print("\nTOP FEATURES")

print(
    feature_importance_summary.head(15).to_string(index=False)
    if not feature_importance_summary.empty
    else "Feature importance unavailable."
)

print("\nPrediction error records:")
print("Validation:", len(validation_errors))
print("Testing:", len(test_errors))

print("\nSaved model directory:", MODEL_DIR)
print("Saved output directory:", OUTPUT_DIR)

print("\nAll machine learning forecasting outputs saved.")


Dataset loaded successfully.
Date column: date
Target column: children_in_hhs_care
Observations: 690

Chronological boundaries:
Training observations: 482
Validation observations: 104
Testing observations: 104

Feature preparation completed.
Number of features: 23
Modeling observations: 660

Forecast horizons: {'1_day': 1, '7_day': 7, '14_day': 14, '30_day': 30}

Training Random_Forest for 1_day
Best parameters: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Best CV MAE: 743.5069279498283

Training Gradient_Boosting for 1_day
Best parameters: {'learning_rate': 0.1, 'max_depth': 2, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best CV MAE: 745.950661280901

Training Random_Forest for 7_day
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best CV MAE: 1252.2229459459459

Training Gradient_Boosting for 7_day
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_sampl